# Personal Research Assistant (RAG)

In [ ]:
import os 
import shutil # it is used to find the path of npx on windows, which is needed to run the MCP server.
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper
from langchain_core.tools import Tool
from langchain_core.tools.retriever import create_retriever_tool # this is a function that allows us to create a retriever tool from a retriever object. 
from langchain_community.tools.openweathermap import OpenWeatherMapQueryRun
from langchain_community.utilities.openweathermap import OpenWeatherMapAPIWrapper
# New imports for MCP
import asyncio
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_core.tools import StructuredTool


In [ ]:
if os.name == 'nt':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    npx_path = shutil.which("npx")

In [ ]:
# function to give ability to clara to connect to MCPs 

def convert_mcp_to_langchain(mcp_tool, session:ClientSession, server_name:str): 
    async def _run_tool(**kwargs): # * kwargs means that we are taking in a variable number of keyword arguments and passing them to the MCP tool. This allows us to have a flexible interface that can work with any MCP tool regardless of the number of arguments it takes.
        # This is the logic that actually calls the external MCP server
        result = await session.call_tool(mcp_tool.name, arguments=kwargs) # here session.call is an readymade method that allows us to call a tool on MCP server by passing in the tool name and the arguments.
        return result.content.text

        return StructuredTool(
        name=f"{server_name}_{mcp_tool.name}", 
        description=mcp_tool.description,
        func=_run_tool, 
        args_schema=mcp_tool.inputSchema
    )

    # here basically we have a function that takes in an MCP tool and converts it into a langchain tool by creating a wrapper function that calls MCP tool
    # the wrapper function is an async function that takes in the same arguments as the MCP tool and then calls the MCP tool using the session.call_tool method and returns the result.
    # then we create a StructuredTool using the wrapper function as the func and the same name and description as the MCP tool.

In [ ]:
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["EXA_API_KEY"] = os.getenv("EXA_API_KEY")
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.9)


In [ ]:
exit_stack = AsyncExitStack()
exa_params = StdioServerParameters(
    command=npx_path,  # Use the full path found by shutil
    args=["-y", "@modelcontextprotocol/server-exa"],
)
print("connecting to mcp server...")

read, write = await exit_stack.enter_async_context(stdio_client(exa_params))
session = await exit_stack.enter_async_context(ClientSession(read, write))
await session.initialize()


In [ ]:
DATA_PATH = "." # here DATA_PATH = . means we have taken root of folder .
# Directory loader is used to load multiple data or documents like pdf 
loader = DirectoryLoader(DATA_PATH, glob="**/*.pdf", show_progress=True, loader_cls=PyPDFLoader)
documents = loader.load()
print(f"Loaded {len(documents)} total pages.")


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(documents)    
print(f"Split into {len(texts)} chunks of text (with overlap).")

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)
vector_store = FAISS.from_documents(texts, embeddings)
vector_store.save_local("faiss_local_db")
load_retriever = vector_store.as_retriever(search_kwargs={"k": 3}) # This tells the database to return the top 3 most relevant chunks when asked a question


In [ ]:
# Weather Tool 
weather_wrapper = OpenWeatherMapAPIWrapper()
weather_tool = OpenWeatherMapQueryRun(api_wrapper=weather_wrapper )
weather_tool.name = "WeatherTool"

In [ ]:
wiki_wrapper = WikipediaAPIWrapper(top_k_results=5, doc_content_chars_max=250) # it means that it will return top 5 results for the query
wiki_tool = WikipediaQueryRun(api_wrapper=wiki_wrapper)
print(f"Tool created: {wiki_tool.name}")
wiki_tool.name = "wikipedia_search"


In [ ]:
arxiv_wrapper = ArxivAPIWrapper(top_k_results=5, doc_content_chars_max=250) # it means that it will return top 5 results for the query
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv_tool.name = "arxiv_search"

print(f"tool created: {arxiv_tool.name}")

In [ ]:
local_pdf_tool = create_retriever_tool(
    retriever=vector_store.as_retriever(),
    name="local_pdf_search",
    description="Search the user's uploaded local PDFs. Always use this first if the user asks about their own documents."
)

In [ ]:
# Tavily Search Tool 
# It is used for web search 

import langchain_community


from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults(description="use this tool to get the latest news and information from the web")
tavily.invoke("latest news on AI")

In [ ]:
# Getting tools from MCP server

mcp_tools_resp = await session.list_tools()
mcp_tools_list = [convert_mcp_to_langchain(t, session, "exa") for t in mcp_tools_resp.tools]

print(f"✅ Loaded {len(mcp_tools_list)} MCP tools!")

In [ ]:
# now we will create a list of tools that we will use in our agent.
tools = [local_pdf_tool, wiki_tool, arxiv_tool, weather_tool, tavily]+ mcp_tools_list
print(f"Tools created: {len(tools)} and made the list of  {[tool.name for tool in tools]}")



In [ ]:
# Till here we have sucessfully followed all 4 steps of rag process till creating the tools. Now we will move to the next step which is creating the agent and giving it access to these tools. We will do that in agents.ipynb file.
from langgraph.prebuilt import create_react_agent
print("Agent created and given access to tools")
agent = create_react_agent(llm , tools) # think create_react_agent in this way: This is a pre-built LangGraph function. It automatically writes a hidden System Prompt for your LLM that says: "You are an AI with access to these specific tools. When you want to use one, reply with a JSON object containing the tool name and your search query. Wait for the system to give you the result before answering."
query = "What is Gen AI ? and give me current weather in delhi and also give me latest news about AI and also tell me if there is any research paper published in arxiv about gen ai in last 1 month and also tell me if there is anything about gen ai in the pdfs that i have uploaded ?"
print("agent starting")
inputs = {"messages": [("user", query)]}

for step in agent.stream(inputs, stream_mode="values"):
    last_message = step["messages"][-1]
    
    # Check if the agent decided to call a tool
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        print(f"---  AGENT DECIDED TO USE TOOL: {last_message.tool_calls[0]['name']} ---")
    
    # Print the final text response from the AI
    elif last_message.type == "ai" and last_message.content:
        print(f"\n FINAL ANSWER --\n{last_message.content}")


